# Daume et al. (2024) on dandiset 000673: LFP phase-amplitude coupling

**Paper.** Daume J, Kaminski J, Schjetnan AGP, Salimpour Y, Khan U, Kyzar M, Reed CM, Anderson WS,
Valiante TA, Mamelak AN, Rutishauser U (2024). Control of working memory by phase-amplitude
coupling of human hippocampal neurons. *Nature* 629, 393-401.
[doi:10.1038/s41586-024-07309-z](https://doi.org/10.1038/s41586-024-07309-z).

**Dandiset.** [DANDI:000673](https://dandiarchive.org/dandiset/000673/0.240118.2135), version
0.240118.2135, the version the paper cites (reference 59); see the header of
`reproduce_daume_2024.ipynb` for the later versions.

**Text the claims were written from.** The version of record (PMC11078732) with its Methods,
legends and Supplementary Information, and the paper's code (SBCAT-release-NWB at a25e8cc3).

This notebook tests the claims on LFP phase-amplitude coupling (PAC). Each channel's LFP is
streamed, cut into -0.5 to 3 s epochs around maintenance onset, filtered trial by trial as the
paper's code does, and Tort's modulation index (MI) is z-scored against 200 trial-shuffled
surrogates (`sbcat.py` says where each parameter comes from). Every result is shown under both
readings of the artefact rejection (O1): (a) on the epochs, (b) on whole trials. Where they
disagree, (a) is adopted: on the one channel whose rejected trials the paper's code ships
(`artif_trials.mat`, sub-26 session 1, channel 33), every trial (a) removes is among the paper's,
while (b) removes 5 trials the paper kept (data check below).

| Claim | Tested here |
|---|---|
| fig2a-theta-gamma-pac | yes |
| fig2b-pac-channel-proportions | yes |
| fig2c-hippocampus-load | yes |
| fig2c-no-load-effect-other-areas | yes |
| edf2b-hippocampus-load-patients | yes |
| edf2c-low-gamma-no-load-effect | yes |
| tables1-slow-fast-theta | yes |
| fig2f-rt-single-trial-pac | yes |
| edf3-hippocampus-pac-controls | in `reproduce_daume_2024_lfp_controls.ipynb` |
| edf3a-kappa-load | yes |
| edf3e-nesting | yes |
| edf3f-raw-mi-load | yes |
| methods-spectral-slope | yes |
| fig3h-pac-category-firing | yes |

In [ ]:
import os
for _k in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"]:
    os.environ[_k] = "1"   # many small fits run in parallel processes
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats

import sbcat

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore")

started = time.time()
S = sbcat.load_all(sbcat.list_assets())
print(f"loaded trials and units of {len(S)} files in {time.time() - started:.0f} s")
t0 = time.time()
R = sbcat.run_all_lfp(S, seed=201, workers=6)
print(f"LFP of {len(R)} sessions, {sum(len(r['channels']) for r in R)} channels, processed in {(time.time() - t0) / 60:.1f} min")

In [ ]:
HG = np.where((sbcat.AMP_CENTRES >= 70) & (sbcat.AMP_CENTRES <= 140))[0]
LG = np.where((sbcat.AMP_CENTRES >= 30) & (sbcat.AMP_CENTRES <= 55))[0]
READINGS = ["a", "b"]


def band_z(zdict, phases, amp_idx):
    """Mean z over the given phase centres and amplitude indices (O3)."""
    return float(np.mean([zdict[p][amp_idx].mean() for p in phases]))


rows = []
for r in R:
    s = S[r["session"]]
    for c in r["channels"]:
        for rd in READINGS:
            o = c["readings"][rd]
            base = {"session": c["session"], "subject": c["subject"], "path": c["path"], "channel": c["channel"],
                    "area": c["area"], "reading": rd, "kept_trials": int(c["masks"][rd].sum()),
                    "n_trials": len(c["masks"][rd])}
            if o is None:
                rows.append({**base, "n_per_load": 0})
                continue
            rows.append({**base, "n_per_load": o["n_per_load"],
                         "z_thetaHG": band_z(o["z_all"], sbcat.THETA, HG), "z_thetaLG": band_z(o["z_all"], sbcat.THETA, LG),
                         "z1_thetaHG": band_z(o["z_l1"], sbcat.THETA, HG), "z3_thetaHG": band_z(o["z_l3"], sbcat.THETA, HG),
                         "z1_thetaLG": band_z(o["z_l1"], sbcat.THETA, LG), "z3_thetaLG": band_z(o["z_l3"], sbcat.THETA, LG),
                         "mi1_thetaHG": band_z(o["mi_l1"], sbcat.THETA, HG), "mi3_thetaHG": band_z(o["mi_l3"], sbcat.THETA, HG),
                         **{f"z_{nm}_{g}": band_z(o["z_all"], ph, ix) for nm, ph in [("slowA", (2, 4)), ("fastA", (6, 8))] for g, ix in [("HG", HG), ("LG", LG)]},
                         **{f"z{L}_{nm}_{g}": band_z(o[f"z_l{L}"], ph, ix) for L in (1, 3) for nm, ph in [("slowA", (2, 4)), ("fastA", (6, 8))] for g, ix in [("HG", HG), ("LG", LG)]},
                         **{f"zb_{k}_{g}_{cn}": v for (k, g, cn), v in o["bands"].items()}})
ch = pd.DataFrame(rows)
chan_index = {(c["session"], c["channel"]): c for r in R for c in r["channels"]}
print(ch.groupby("reading").size().to_dict())

## Data check

In [ ]:
one = ch[ch.reading == "a"]
print("LFP channels by area:", one.area.value_counts().reindex(sbcat.AREAS).to_dict(), "; total", len(one))
info = pd.DataFrame([{"path": r["path"], "lfp_samples": r["lfp_shape"][0], "channels": r["lfp_shape"][1],
                      "lfp_s": r["lfp_shape"][0] / 400, "task_end_s": S[r["session"]]["trials"].stop_time.max()} for r in R])
print("LFP length minus the end of the last trial (s): min %.1f, max %.1f" % ((info.lfp_s - info.task_end_s).min(), (info.lfp_s - info.task_end_s).max()))
print("LFP rate and unit as stored:", {(s["lfp_info"]["rate"], s["lfp_info"]["unit"], s["lfp_info"]["conversion"]) for s in S if s["lfp_info"]})
sd = pd.Series([c["sd_uV"] for r in R for c in r["channels"]])
p99 = pd.Series([c["p99_abs_uV"] for r in R for c in r["channels"]])
print("per-channel s.d. of the epochs (stored units): median %.1f, range %.2f to %.1f; 99th percentile of |LFP|: median %.1f, max %.1f" % (
    sd.median(), sd.min(), sd.max(), p99.median(), p99.max()))
for rd in READINGS:
    d = ch[ch.reading == rd]
    rem = d.n_trials - d.kept_trials
    print(f"reading {rd}: trials removed per channel {rem.mean():.1f} +/- {rem.std():.1f} ({100 * rem.sum() / d.n_trials.sum():.1f}%); "
          f"trials per load after subsampling: median {d.n_per_load.median():.0f}, range {d.n_per_load.min()} to {d.n_per_load.max()}; "
          f"channels with fewer than 3 per load (left out): {int((d.n_per_load < 3).sum())}")
print("paper: 20.4 +/- 13.9 trials (14.6%) removed per wire, with visual inspection")
fp = pd.Series({r["path"]: r["lfp_fingerprint"] for r in R})
print("files sharing the sum of their first 4,000 LFP samples (duplicates):", fp[fp.duplicated(keep=False)].to_dict() or "none")
for rd in READINGS:
    d = ch[ch.reading == rd]
    rem = d.n_trials - d.kept_trials
    print(f"reading {rd}: trials removed per channel, percentiles 0/25/50/75/90/100: {np.percentile(rem, [0, 25, 50, 75, 90, 100]).round(1).tolist()}; "
          f"channels with none removed {int((rem == 0).sum())} of {len(rem)}; histogram (0-4, 5-9, ..., 50+): "
          f"{np.bincount(np.minimum(rem.to_numpy().astype(int) // 5, 10), minlength=11).tolist()}")

In [ ]:
# The paper's code (NWB_samplePAC_SU.m, SBCAT-release-NWB at a25e8cc3) ships the artefact trials of
# its example: sub-26, session 1, LFP channel 33 (1-based), in artif_trials.mat. Compare them with the
# trials each reading of O1 removes on that channel.
import io
import urllib.request
from scipy.io import loadmat
url = ("https://raw.githubusercontent.com/rutishauserlab/SBCAT-release-NWB/a25e8cc3f61630a428f190381745257d4506f148/"
       "NWB_SBCAT_analysis/helpers/internal/PAC/artif_trials.mat")
with urllib.request.urlopen(url) as r_:
    paper_artif = set(int(v) - 1 for v in loadmat(io.BytesIO(r_.read()))["artif_trials"].ravel())
si26 = [i for i, s in enumerate(S) if "sub-26_ses-1_" in s["path"]]
print("sub-26 session 1:", [S[i]["path"] for i in si26])
c26 = chan_index[(si26[0], 32)]
print(f"channel 33 ({c26['location']}): the paper's code removes {len(paper_artif)} trials")
for rd in READINGS:
    rem = set(np.where(~c26["masks"][rd])[0])
    print(f"  reading {rd} removes {len(rem)}; {len(rem & paper_artif)} of them among the paper's {len(paper_artif)}, "
          f"{len(rem - paper_artif)} that the paper kept")

In [ ]:
# Clock check by contrast: RMS of the probe-locked trial-averaged LFP 0.1 to 0.6 s after onset over
# that of -0.6 to -0.1 s, at the true probe onsets and at onsets moved 1 or 2 s back (into maintenance).
ev = pd.DataFrame([{**{f"shift_{k}": v for k, v in c["evoked"].items()}, "path": c["path"]} for r in R for c in r["channels"]])
print(ev.drop(columns="path").median().round(2).to_dict())
per_sess = ev.groupby("path").median()
print("sessions whose median ratio is larger at the true onset than at both shifts:",
      int(((per_sess["shift_0.0"] > per_sess["shift_-1.0"]) & (per_sess["shift_0.0"] > per_sess["shift_-2.0"])).sum()), "of", len(per_sess))
print("sessions where it is not:", per_sess.index[~((per_sess["shift_0.0"] > per_sess["shift_-1.0"]) & (per_sess["shift_0.0"] > per_sess["shift_-2.0"]))].tolist())

## fig2a-theta-gamma-pac

> Across all recorded channels, PAC during maintenance was strongest between theta phase (3-7 Hz)
> and the amplitude of a low (30-55 Hz) and a high (70-140 Hz) gamma band.

In [ ]:
for rd in READINGS:
    PH = sbcat.PHASE_CENTRES if rd == "a" else np.array([2, 4, 6, 8])   # reading (b) computes phase 2-8 Hz only
    Z = np.array([[c["readings"][rd]["z_all"][int(p)] for p in PH] for r in R for c in r["channels"] if c["readings"][rd] is not None])
    mean_z = Z.mean(0)
    print(f"reading {rd}: {len(Z)} channels; mean z comodulogram (rows phase {PH[0]}-{PH[-1]} Hz, columns amplitude 30-150 Hz):")
    print(pd.DataFrame(mean_z, index=PH, columns=sbcat.AMP_CENTRES).round(2).to_string())
    ip, ia = np.unravel_index(np.argmax(mean_z), mean_z.shape)
    print(f"largest mean z {mean_z[ip, ia]:.2f} at phase {PH[ip]} Hz, amplitude {sbcat.AMP_CENTRES[ia]} Hz")
    for p in sbcat.THETA:
        prof = mean_z[list(PH).index(p)]
        peaks = [int(sbcat.AMP_CENTRES[i]) for i in range(1, len(prof) - 1) if prof[i] > prof[i - 1] and prof[i] > prof[i + 1]]
        print(f"phase {p} Hz: local maxima of the amplitude profile at {peaks} Hz")
    by_area = {a: Z[[ (c['area'] == a) for r in R for c in r['channels'] if c['readings'][rd] is not None]].mean(0) for a in sbcat.AREAS}
    print("largest mean z per area (phase, amplitude):", {a: (int(PH[np.unravel_index(m.argmax(), m.shape)[0]]), int(sbcat.AMP_CENTRES[np.unravel_index(m.argmax(), m.shape)[1]]), round(float(m.max()), 2)) for a, m in by_area.items()})
    fig, ax = plt.subplots(figsize=(6, 3))
    im = ax.imshow(mean_z, origin="lower", aspect="auto", extent=[27.5, 152.5, PH[0] - 1, PH[-1] + 1], cmap="viridis")
    ax.set_xlabel("amplitude frequency (Hz)"); ax.set_ylabel("phase frequency (Hz)"); plt.colorbar(im, label="mean z (MI)")
    ax.set_title(f"all channels, reading {rd}; range {mean_z.min():.2f} to {mean_z.max():.2f}")
    plt.show()

**Result: reproduced.** Under reading (a), over 1,922 channels, the largest mean z (2.16) is at
phase 4 Hz and amplitude 80 Hz; at 4 Hz the amplitude profile has local maxima at 50 Hz (within 30
to 55) and 80 Hz (within 70 to 140). Under reading (b), computed for phase centres 2 to 8 Hz over
1,918 channels, the largest mean z (1.73) is again at 4 Hz and 80 Hz, with local maxima at 45 and
80 Hz. Both readings meet the criterion.

## fig2b-pac-channel-proportions

> 137 of 586 hippocampal channels showed significant theta-high-gamma PAC; the proportions of
> such channels in the hippocampus, amygdala and vmPFC exceeded the trial-shuffled null, while
> pre-SMA and dACC had few such channels.

In [ ]:
paper_counts = {"hippocampus": 137, "amygdala": 130, "preSMA": 5, "dACC": 14, "vmPFC": 40}
fig2b_rows = []
for rd in READINGS:
    for a in sbcat.AREAS:
        d = ch[(ch.reading == rd) & (ch.area == a) & (ch.n_per_load >= 3)]
        sig = d.z_thetaHG > 1.64
        surr = np.array([chan_index[(r.session, r.channel)]["readings"][rd]["surr_thetaHG"] for r in d.itertuples()])
        null = (surr > 1.64).mean(0)            # proportion of channels per surrogate (O4)
        prop = sig.mean()
        fig2b_rows.append({"reading": rd, "area": a, "channels": len(d), "significant": int(sig.sum()), "paper": paper_counts[a],
                           "within_10pct": abs(sig.sum() - paper_counts[a]) <= 0.1 * paper_counts[a],
                           "proportion": prop, "null_mean": null.mean(), "null_p99": np.percentile(null, 99),
                           "p": (np.sum(null >= prop) + 1) / (len(null) + 1)})
fig2b = pd.DataFrame(fig2b_rows)
print(fig2b.round(4).to_string(index=False))
sig_a = ch[(ch.reading == "a") & (ch.z_thetaHG > 1.64)]
print("\nsignificant hippocampal channels per patient (reading a):", sig_a[sig_a.area == "hippocampus"].groupby("subject").size().to_dict())
for thr in [1.28, 1.64, 1.96, 2.33]:
    print(f"threshold z > {thr}:", ch[(ch.reading == 'a') & (ch.z_thetaHG > thr)].area.value_counts().reindex(sbcat.AREAS).to_dict())

**Result: partially reproduced.** Under reading (a), 144 hippocampal (paper 137), 130 amygdala
(130), 41 vmPFC (40) and 14 dACC (14) channels are significant, all within tolerance, but the
pre-SMA has 17 against 5. Every area's proportion is above the 99th percentile of its null (p =
0.005, the smallest possible with 200 surrogates), the pre-SMA and dACC too. Under reading (b),
the amygdala (112), dACC (8), vmPFC (27) and pre-SMA (12) counts are out of tolerance.

## fig2c-hippocampus-load and fig2c-no-load-effect-other-areas

> In the 137 hippocampal channels with significant PAC, theta-high-gamma PAC was weaker in load 3
> than in load 1 trials.

> Theta-high-gamma PAC did not differ between loads in the amygdala, vmPFC, pre-SMA or dACC
> channels with significant PAC.

In [ ]:
rng = np.random.default_rng(301)
fig2c_rows = []
for rd in READINGS:
    for a in sbcat.AREAS:
        d = ch[(ch.reading == rd) & (ch.area == a) & (ch.z_thetaHG > 1.64)]
        if len(d) < 2:
            fig2c_rows.append({"reading": rd, "area": a, "n": len(d)})
            continue
        t = sbcat.perm_ttest_paired(d.z3_thetaHG, d.z1_thetaHG, rng)
        fig2c_rows.append({"reading": rd, "area": a, "n": t["n"], "load1": t["mean_y"], "load3": t["mean_x"],
                           "t(3-1)": t["t"], "p": t["p"], "channels_load1_higher": int((d.z1_thetaHG > d.z3_thetaHG).sum())})
fig2c = pd.DataFrame(fig2c_rows)
fig2c["p_fdr"] = np.nan
for rd in READINGS:
    m = (fig2c.reading == rd) & fig2c.p.notna()
    fig2c.loc[m, "p_fdr"] = sbcat.fdr_bh(fig2c.loc[m, "p"])
print(fig2c.round(4).to_string(index=False))
d = ch[(ch.reading == "a") & (ch.area == "hippocampus") & (ch.z_thetaHG > 1.64)]
pp = d.assign(diff=d.z1_thetaHG - d.z3_thetaHG).groupby("subject")["diff"].mean()
print("\nhippocampus, reading a: patients with load 1 above load 3 (mean over their channels):", int((pp > 0).sum()), "of", len(pp))
print("range of the channel z values (reading a, hippocampus): load 1 %.2f to %.2f, load 3 %.2f to %.2f" % (d.z1_thetaHG.min(), d.z1_thetaHG.max(), d.z3_thetaHG.min(), d.z3_thetaHG.max()))

In [ ]:
print(fig2c[fig2c.area != "hippocampus"].round(4).to_string(index=False))
print("paper: amygdala t129 = 1.43, p = 0.38; vmPFC t39 = 0.16, p = 0.87; pre-SMA t4 = -0.16, p = 0.87; dACC t13 = -0.82, p = 0.73")
for rd in READINGS:
    d = ch[(ch.reading == rd) & (ch.area == "preSMA") & (ch.z_thetaHG > 1.64)]
    print(f"\npre-SMA PAC channels, reading {rd} (z of theta-high-gamma MI per load):")
    print(d[["subject", "session", "channel", "z1_thetaHG", "z3_thetaHG"]].assign(diff=d.z3_thetaHG - d.z1_thetaHG).round(2).to_string(index=False))
    r_ = sbcat.perm_ttest_paired(d.z3_thetaHG, d.z1_thetaHG, np.random.default_rng(310))
    w = stats.wilcoxon(d.z3_thetaHG, d.z1_thetaHG)
    top = d.assign(diff=d.z3_thetaHG - d.z1_thetaHG).sort_values("diff").iloc[:-3]
    r3 = sbcat.perm_ttest_paired(top.z3_thetaHG, top.z1_thetaHG, np.random.default_rng(311))
    print(f"load 3 higher in {int((d.z3_thetaHG > d.z1_thetaHG).sum())} of {len(d)}; paired permutation t-test p = {r_['p']:.4f}; "
          f"Wilcoxon signed-rank p = {w.pvalue:.4f}; without the 3 channels of largest load 3 excess, t = {r3['t']:.2f}, p = {r3['p']:.4f}")

**Result: fig2c-hippocampus-load reproduced.** Hippocampal PAC is lower in load 3 than in load 1
under both readings (a: 144 channels, t = -4.16, FDR p = 0.0010; b: 125 channels, t = -3.05, FDR p
= 0.014).

**Result: fig2c-no-load-effect-other-areas not reproduced.** Under reading (a) the pre-SMA (17
channels, load 3 higher, FDR p = 0.0017) and the amygdala (130 channels, load 3 higher, FDR p =
0.0475) differ between loads; under (b) the pre-SMA (FDR p = 0.014) and the dACC (8 channels,
load 1 higher, FDR p = 0.014) do. The vmPFC does not differ under either (p = 0.93, 0.94). The
pre-SMA channels come from six patients; four channels of sub-4 have z values of 19
to 25 in load 3 against 3.5 to 6.8 in load 1, and seven of sub-11 are also higher in load
3. Load 3 is higher in 14 of the 17 channels (Wilcoxon p = 0.0011), and without the three channels
of largest excess the test still gives p = 0.0045 under reading (a); under (b) it gives p = 0.059.

## edf2b-hippocampus-load-patients

> Averaging significant PAC channels within each patient, hippocampal PAC was again stronger in
> load 1 than in load 3, and no other area showed a load difference.

In [ ]:
rng = np.random.default_rng(302)
rows = []
for rd in READINGS:
    for a in sbcat.AREAS:
        d = ch[(ch.reading == rd) & (ch.area == a) & (ch.z_thetaHG > 1.64)]
        pm = d.groupby("subject")[["z1_thetaHG", "z3_thetaHG"]].mean()
        if len(pm) < 2:
            rows.append({"reading": rd, "area": a, "patients": len(pm)})
            continue
        t = sbcat.perm_ttest_paired(pm.z1_thetaHG, pm.z3_thetaHG, rng)
        rows.append({"reading": rd, "area": a, "patients": len(pm), "load1": t["mean_x"], "load3": t["mean_y"], "t(1-3)": t["t"], "p": t["p"],
                     "patients_load1_higher": int((pm.z1_thetaHG > pm.z3_thetaHG).sum())})
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("paper: hippocampus n = 23 patients, p = 0.0049")

**Result: reproduced, resting on an assumption.** Under reading (a), 23 patients have hippocampal
PAC channels (paper 23), their means are higher in load 1 (3.97 against 3.23, p = 0.0095; 18 of 23
patients), and no other area differs (p >= 0.10). Under reading (b) the hippocampus gives p = 0.11
over 20 patients. Reading (a) is adopted (header and data check: the paper's own rejected trials
on its example channel include every trial (a) removes).

## edf2c-low-gamma-no-load-effect

> Theta-low-gamma (30-55 Hz) PAC did not differ between loads in any region.

In [ ]:
rng = np.random.default_rng(303)
rows = []
for rd in READINGS:
    for a in sbcat.AREAS:
        d = ch[(ch.reading == rd) & (ch.area == a) & (ch.z_thetaLG > 1.64)]
        if len(d) < 2:
            rows.append({"reading": rd, "area": a, "n": len(d)})
            continue
        t = sbcat.perm_ttest_paired(d.z3_thetaLG, d.z1_thetaLG, rng)
        rows.append({"reading": rd, "area": a, "n": t["n"], "load1": t["mean_y"], "load3": t["mean_x"], "t(3-1)": t["t"], "p": t["p"]})
lg = pd.DataFrame(rows)
lg["p_fdr"] = np.nan
for rd in READINGS:
    m = (lg.reading == rd) & lg.p.notna()
    lg.loc[m, "p_fdr"] = sbcat.fdr_bh(lg.loc[m, "p"])
print(lg.round(4).to_string(index=False))
print("paper n: 148, 155, 10, 22 and 48 channels (hippocampus, amygdala, pre-SMA, dACC, vmPFC)")

**Result: not reproduced.** The pre-SMA's low-gamma PAC differs between loads under both readings
(a: 22 channels, p = 0.0029; b: 13 channels, p = 0.0033, load 3 higher); no other area does (p >=
0.11). The counts of selected channels are 152, 159, 22, 27 and 48 under (a), against the paper's
148, 155, 10, 22 and 48.

## tables1-slow-fast-theta

> Split into slow (2-5 Hz) and fast (5-9 Hz) theta, PAC differed between loads only in the
> hippocampus and only with high gamma, most strongly for fast theta (t116 = -3.05, p = 0.0015;
> slow theta, t147 = -2.31, p = 0.0163).

Readings: (a) phase centres 2 and 4 Hz (slow) and 6 and 8 Hz (fast) of the comodulogram; (b)
bands filtered at 2-5 and 5-9 Hz. Each under both artefact readings.

In [ ]:
rng = np.random.default_rng(304)
rows = []
for rd in READINGS:
    for band_reading in ["A", "B"]:
        for th in ["slow", "fast"]:
            for g in ["HG", "LG"]:
                if band_reading == "A":
                    sel_col, c1, c3 = f"z_{th}A_{g}", f"z1_{th}A_{g}", f"z3_{th}A_{g}"
                else:
                    sel_col, c1, c3 = f"zb_{th}_{g}_all", f"zb_{th}_{g}_l1", f"zb_{th}_{g}_l3"
                for a in sbcat.AREAS:
                    d = ch[(ch.reading == rd) & (ch.area == a) & (ch[sel_col] > 1.64)]
                    if len(d) < 2:
                        rows.append({"artefact": rd, "bands": band_reading, "theta": th, "gamma": g, "area": a, "n": len(d)})
                        continue
                    t = sbcat.perm_ttest_paired(d[c3], d[c1], rng)
                    rows.append({"artefact": rd, "bands": band_reading, "theta": th, "gamma": g, "area": a, "n": t["n"],
                                 "load1": t["mean_y"], "load3": t["mean_x"], "t(3-1)": t["t"], "p": t["p"]})
ts1 = pd.DataFrame(rows)
print(ts1.round(4).to_string(index=False))
summ = ts1.assign(sig=ts1.p < 0.05).groupby(["artefact", "bands"]).apply(
    lambda g: pd.Series({"tests": int(g.p.notna().sum()), "significant": int(g.sig.sum()),
                         "significant_tests": "; ".join(f"{r.area} {r.theta} {r.gamma} t={r['t(3-1)']:.2f}" for _, r in g[g.sig].iterrows())}))
print(summ.to_string())

**Result: not reproduced.** Under every combination of readings both hippocampal high-gamma tests
give p < 0.05 with load 1 higher, but other tests do too: 8, 5, 6 and 6 of 20 tests are
significant, always including pre-SMA tests with load 3 higher, and under (a, A) also the
hippocampal slow theta to low gamma test.

## fig2f-rt-single-trial-pac

> In the hippocampus, stronger single-trial theta-high-gamma PAC went with faster RTs, while no
> such relation held in the amygdala or vmPFC.

Single-trial PAC (O5): (a) the MI of each trial's 0 to 2.5 s window, averaged over the theta and
high-gamma pairs, raw; (b) the same z-scored across trials within each channel. Correct, clean
trials of the significant channels; RT ~ 1 + PAC * load + (1 | patient) + (1 | channel nested in
patient), normal response (Supplementary Table S2), fitted by REML with statsmodels MixedLM. Load
is coded with load 3 as the reference, as Table S2 names the terms "Load1" and "PAC : Load1" and
says "Load 1 are tested against load 3 trials"; the PAC term is then the slope in load 3 trials.
The slopes in load 1 trials and averaged over loads are also printed. Table S2 gives no unit or
transform for RT, and its Load1 term (-0.48) cannot be a difference of raw RTs in s or ms, so RT
is fitted in s, as log10 RT, and z-scored over each session's correct trials.

In [ ]:
def slopes(m, term, ind="load1"):
    """The term's slope in load 3 trials, in load 1 trials and averaged over the loads, with Wald z and p.
    load1 is an indicator of load 1 trials, so load 3 is the reference: the paper's coding, as Tables
    S2 to S4 name the terms "Load1" and "PAC : Load1" and say "Load 1 are tested against load 3 trials".
    The model's own PAC (or SC) term is then the load 3 slope."""
    V = m.cov_params()
    b, i = term, f"{term}:{ind}"
    out = {}
    for name, w in [("L3", 0.0), ("L1", 1.0), ("mean", 0.5)]:
        est = m.params[b] + w * m.params[i]
        z = est / np.sqrt(V.loc[b, b] + w * w * V.loc[i, i] + 2 * w * V.loc[b, i])
        out.update({f"{name} est": est, f"{name} t": z, f"{name} p": 2 * stats.norm.sf(abs(z))})
    out.update({"Load1 est": m.params[ind], "int est": m.params[i], "int p": m.pvalues[i], "converged": m.converged})
    return out


def rt_frame(rd, a):
    """Correct, clean trials of the area's significant PAC channels under artefact reading rd. RT in s,
    log10 RT, and RT z-scored over the session's correct trials (the paper gives no unit or transform)."""
    d = ch[(ch.reading == rd) & (ch.area == a) & (ch.z_thetaHG > 1.64)]
    parts = []
    for r in d.itertuples():
        c = chan_index[(r.session, r.channel)]
        tr = S[r.session]["trials"]
        rt_c = tr.rt[tr.correct]
        idx = c["trial_index"]
        keep = c["masks"][rd][idx]
        pac = c["st_thetaHG"][keep]
        rt = tr.rt.to_numpy()[idx][keep]
        parts.append(pd.DataFrame({"rt": rt, "log_rt": np.log10(rt), "z_rt": (rt - rt_c.mean()) / rt_c.std(),
                                   "load1": (tr.load.to_numpy()[idx][keep] == 1).astype(int),
                                   "pac_a": pac, "pac_b": (pac - np.nanmean(pac)) / np.nanstd(pac),
                                   "patient": r.subject, "channel": f"{r.session}_{r.channel}"}))
    return d.shape[0], pd.concat(parts).dropna().reset_index(drop=True)


glm_rows, frames = [], {}
for rd in READINGS:
    for a in ["hippocampus", "amygdala", "vmPFC"]:
        n_ch, df = rt_frame(rd, a)
        frames[(rd, a)] = df
        for resp in ["rt", "log_rt", "z_rt"]:
            for o5 in ["a", "b"]:
                base = {"artefact": rd, "O5": o5, "response": resp, "area": a, "channels": n_ch, "trials": len(df)}
                try:
                    m = smf.mixedlm(f"{resp} ~ pac_{o5} * load1", df, groups="patient",
                                    vc_formula={"channel": "0 + C(channel)"}).fit(reml=True)
                    glm_rows.append({**base, "intercept": m.params["Intercept"], **slopes(m, f"pac_{o5}")})
                except Exception as e:  # report a failed fit
                    glm_rows.append({**base, "error": str(e)[:80]})
glm = pd.DataFrame(glm_rows)
fmt = lambda v: f"{v:.4g}"
print("PAC slope in load 3 trials (L3, the paper's PAC term), in load 1 trials (L1) and averaged (mean);")
print("Load1 est is the load 1 minus load 3 difference at PAC = 0, int the PAC x Load1 term; intercept is load 3 at PAC = 0.")
print(glm.to_string(index=False, float_format=fmt))
print("paper (Table S2): hippocampus PAC -49.07, t = -4.01, p = 6e-5, Load1 -0.48, PAC:Load1 26.30 (p = 0.11), intercept 0.38;")
print("amygdala PAC -5.15, p = 0.48, Load1 -0.48; vmPFC PAC 24.51, p = 0.24, Load1 -0.34")
dh = frames[("a", "hippocampus")]
print("RT of the hippocampal trials under artefact reading (a): %d trials, median %.2f s, max %.1f s, %d above 10 s"
      % (len(dh), dh.rt.median(), dh.rt.max(), (dh.rt >= 10).sum()))
for resp in ["rt", "log_rt", "z_rt"]:
    print(f"  {resp}: load 1 mean {dh[dh.load1 == 1][resp].mean():.3f}, load 3 mean {dh[dh.load1 == 0][resp].mean():.3f}")
# Variant on artefact reading (a): RTs below 10 s only (the paper states no trimming)
var_rows = []
for a in ["hippocampus", "amygdala", "vmPFC"]:
    dv = frames[("a", a)]
    dv = dv[dv.rt < 10].reset_index(drop=True)
    for o5 in ["a", "b"]:
        m = smf.mixedlm(f"rt ~ pac_{o5} * load1", dv, groups="patient", vc_formula={"channel": "0 + C(channel)"}).fit(reml=True)
        var_rows.append({"variant": "RT < 10 s", "O5": o5, "area": a, "trials": len(dv), **slopes(m, f"pac_{o5}")})
print("\nartefact reading (a), RT below 10 s:")
print(pd.DataFrame(var_rows).to_string(index=False, float_format=fmt))


**Result: not reproduced (the verified outcome), with a change proposed.** With load 3 as the
reference level, as Table S2 codes load, the PAC term is the slope in load 3 trials. With RT in
s under artefact reading (a), neither single-trial reading meets the criterion: under O5 (a) the
hippocampal term is negative (-53.0, t = -2.43, p = 0.015) but so is the amygdala term (p =
0.027), and under O5 (b) the hippocampal term has p = 0.099. RT in s does not fit Table S2: the
intercept is 1.70 (O5 a) against the paper's 0.38. With RT z-scored over each session's correct
trials the intercept is 0.28 and the Load1 term -0.32 (paper 0.38 and -0.48), and under O5 (a)
the hippocampal term is -49.39, t = -4.19, p = 2.8e-5 (paper -49.07, t = -4.01, p = 6e-5), but
the amygdala term is also negative (p = 0.015, paper p = 0.48); under O5 (b) the hippocampal
term is negative (p = 0.029) and the amygdala (p = 0.20) and vmPFC (p = 0.58) terms are not
significant, which meets the criterion. log10 RT gives the same split (O5 a, amygdala p = 0.036;
O5 b, hippocampus p = 0.0045, amygdala 0.22, vmPFC 0.87). So on the scale that fits Table S2
the two readings of O5 disagree. With RTs below 10 s (in s), both readings meet the criterion
(hippocampus p = 2.8e-7 and 0.00046, amygdala 0.097 and 0.097, vmPFC 0.99 and 0.61). Under
artefact reading (b), every response scale and both readings meet it (hippocampus p <= 0.031,
amygdala p >= 0.48, vmPFC p >= 0.26). The slope in load 1 trials, which the earlier coding
tested, is not significant in the hippocampus with RT in s (p = 0.25 and 0.64).

## Hippocampal PAC channels: kappa, raw MI, nesting, spectral slope and controls

In [ ]:
def hpc_sig(rd):
    return ch[(ch.reading == rd) & (ch.area == "hippocampus") & (ch.z_thetaHG > 1.64)]

### edf3a-kappa-load

> The distribution of gamma amplitude over theta phase was wider (lower kappa) in load 3 than in
> load 1 across hippocampal PAC channels.

In [ ]:
rng = np.random.default_rng(305)
for rd in READINGS:
    d = hpc_sig(rd)
    k1 = np.array([sbcat.circ_kappa_weighted(chan_index[(r.session, r.channel)]["readings"][rd]["P_thetaHG_l1"]) for r in d.itertuples()])
    k3 = np.array([sbcat.circ_kappa_weighted(chan_index[(r.session, r.channel)]["readings"][rd]["P_thetaHG_l3"]) for r in d.itertuples()])
    t = sbcat.perm_ttest_paired(k3, k1, rng)
    print(f"reading {rd}: n = {t['n']}, kappa load 1 {k1.mean():.4f}, load 3 {k3.mean():.4f}, t(3-1) = {t['t']:.2f}, p = {t['p']:.4f}; "
          f"channels with lower kappa in load 3: {int((k3 < k1).sum())}; kappa range {min(k1.min(), k3.min()):.4f} to {max(k1.max(), k3.max()):.4f}")
print("paper: t(136) = -3.7453, p = 0.0001")

**Result: reproduced.** Kappa is lower in load 3 under both readings (a: 144 channels, t = -4.54, p
= 0.0001; b: 125 channels, t = -3.11, p = 0.0024).

### edf3f-raw-mi-load

> Raw (not z-scored) theta-gamma MI was also larger in load 1 than in load 3 in the hippocampal
> PAC channels.

In [ ]:
rng = np.random.default_rng(306)
for rd in READINGS:
    d = hpc_sig(rd)
    t = sbcat.perm_ttest_paired(d.mi3_thetaHG, d.mi1_thetaHG, rng)
    print(f"reading {rd}: n = {t['n']}, raw MI load 1 {t['mean_y']:.2e}, load 3 {t['mean_x']:.2e}, t(3-1) = {t['t']:.2f}, p = {t['p']:.4f}; "
          f"channels with load 1 higher: {int((d.mi1_thetaHG > d.mi3_thetaHG).sum())}")
print("paper: n = 137; t(136) = -4.0264, p = 0.0001")

**Result: reproduced, resting on an assumption.** Raw MI is larger in load 1 under reading (a) (t =
-4.03, p = 0.0002, 99 of 144 channels) but not under (b) (p = 0.066). Reading (a) is adopted.

### edf3e-nesting

> 110 of the 137 hippocampal PAC channels (80.29%) showed theta-high-gamma nesting, and PAC
> remained lower in load 3 than load 1 in those channels.

In [ ]:
rng = np.random.default_rng(307)
print("Nesting readings: 'amp' aligns windows on the maximum of gamma amplitude (the Methods' words),")
print("'peak' on the maximum of the gamma-filtered signal (a gamma peak).")
for rd in READINGS:
    d = hpc_sig(rd)
    allh = ch[(ch.reading == rd) & (ch.area == "hippocampus") & (ch.n_per_load >= 3)]
    for nr in ["amp", "peak"]:
        nest = np.array([chan_index[(r.session, r.channel)]["readings"][rd]["nesting"][nr]["nested"] for r in d.itertuples()])
        nmax = np.array([chan_index[(r.session, r.channel)]["readings"][rd]["nesting"][nr]["n_maxima"] for r in d.itertuples()])
        dn = d[nest]
        t = sbcat.perm_ttest_paired(dn.z3_thetaHG, dn.z1_thetaHG, rng) if len(dn) > 1 else {"t": np.nan, "p": np.nan, "mean_x": np.nan, "mean_y": np.nan}
        nest_all = np.array([chan_index[(r.session, r.channel)]["readings"][rd]["nesting"][nr]["nested"] for r in allh.itertuples()])
        print(f"artefact {rd}, nesting {nr}: nested {nest.sum()} of {len(d)} PAC channels ({100 * nest.mean():.2f}%), "
              f"{100 * nest_all.mean():.1f}% of all {len(allh)} hippocampal channels; channels by number of local maxima {np.bincount(nmax).tolist()}; "
              f"load test in nested channels t(3-1) = {t['t']:.2f}, p = {t['p']:.4f}, load 1 {t['mean_y']:.2f}, load 3 {t['mean_x']:.2f}")
print("paper: 110 of 137 (80.29%); t(109) = -4.10, p = 0.0001")
# How the count depends on the window for local maxima (45 ms in the Methods), reading (a)
d = hpc_sig("a")
rows_ = []
for nr in ["amp", "peak"]:
    avgs = [chan_index[(r.session, r.channel)]["readings"]["a"]["nesting"][nr]["avg"] for r in d.itertuples()]
    for w in [5, 7, 9, 11, 13, 17]:
        nmax = np.array([sbcat.local_maxima(a_, w) for a_ in avgs])
        rows_.append({"alignment": nr, "window_ms": round(2 * w * 2.5, 1), "nested (>= 3 maxima)": int((nmax >= 3).sum()), "of": len(d)})
print(pd.DataFrame(rows_).pivot(index="window_ms", columns="alignment", values="nested (>= 3 maxima)").to_string())

**Result: reproduced, resting on an assumption.** Aligned on gamma peaks, 105 of 144 PAC channels
(72.92%) are nested under reading (a), within tolerance of 110, and PAC is lower in load 3 in them
(t = -3.61, p = 0.0002). Aligned on gamma amplitude maxima, as the Methods' words read, only 9 are
nested (6.25%) and the load test fails (p = 0.77). The count under the peak alignment is 104 to 114
for windows of 35 to 85 ms (30 for 25 ms), and 4 to 15 under the amplitude alignment for every
window, so the outcome rests on the alignment and not on the window. The alignment on gamma peaks
is adopted (see the attempt record).

### methods-spectral-slope

> The slope of the log-log power spectrum did not differ between load 1 and load 3 trials in
> hippocampal channels.

Welch spectrum (1 s Hann windows, 50% overlap; a standard choice) of the 0 to 2.5 s window of the
subsampled trials, averaged per load; linear fit of log10 power on log10 frequency over (a) 2 to
150 Hz, (b) 30 to 150 Hz, leaving out 58-62 and 118-122 Hz (line-noise band-stops, a standard
choice).

In [ ]:
rng = np.random.default_rng(308)
freqs = R[0]["freqs"]
for rd in READINGS:
    d = ch[(ch.reading == rd) & (ch.area == "hippocampus") & (ch.n_per_load >= 3)]
    for fr_name, (f0, f1) in [("2-150 Hz", (2, 150)), ("30-150 Hz", (30, 150))]:
        fm = (freqs >= f0) & (freqs <= f1) & ~((freqs >= 58) & (freqs <= 62)) & ~((freqs >= 118) & (freqs <= 122))
        sl = {}
        for L in (1, 3):
            sl[L] = np.array([np.polyfit(np.log10(freqs[fm]), np.log10(chan_index[(r.session, r.channel)]["readings"][rd][f"psd_l{L}"][fm]), 1)[0]
                              for r in d.itertuples()])
        t = sbcat.perm_ttest_paired(sl[1], sl[3], rng)
        print(f"artefact {rd}, {fr_name}: n = {t['n']}, slope load 1 {sl[1].mean():.4f} +/- {sl[1].std(ddof=1):.4f}, "
              f"load 3 {sl[3].mean():.4f} +/- {sl[3].std(ddof=1):.4f}, t(1-3) = {t['t']:.2f}, p = {t['p']:.4f}")
print("paper: n = 586; -1.7526 +/- 0.3902 against -1.7517 +/- 0.3928, t585 = -0.86, P = 0.39")

**Result: not reproduced, resting on an assumption.** Over 586 hippocampal channels, the slope from
2 to 150 Hz is -1.7260 in load 1 and -1.7300 in load 3 (within tolerance of the paper's values),
and the paired test finds a difference (artefact reading a: t = 2.66, p = 0.0073; b: p = 0.0018).
From 30 to 150 Hz it does not (p = 0.12 and 0.28). The 2 to 150 Hz range, whose mean slope is
closer to the paper's, is adopted (see the attempt record).

## fig3h-pac-category-firing

> Trial by trial, single-trial theta-high-gamma PAC was weakly positively related to the spike
> count of hippocampal category neurons, but not of amygdala category neurons.

Category neurons are selected as in `reproduce_daume_2024.ipynb` (same function and seed, so the
same 94 hippocampal and 181 amygdala neurons). Each is paired with every significant PAC channel
of its bundle (the electrodes of a session with the same location and coordinates; the files name
no bundle). Per pair, the correct trials clean for the channel; SC is the spike count 0 to 2.5 s
after maintenance onset; single-trial PAC under O5 (a) raw and (b) z-scored within the channel.
The model of Table S3, PAC ~ 1 + SC + load + SC x load + (1 | patient) + (1 | patient:pair),
fitted with statsmodels MixedLM (REML) in place of fitglme, with load 3 as the reference level
(Table S3, "Load 1 are tested against load 3 trials"), so the SC term is the slope in load 3
trials; the load 1 and average slopes are also printed. Under the trial readings of the
criterion: (a) all correct trials, (b) correct preferred trials. Artefact reading (a) of O1 is the
one adopted; reading (b) is also shown.

In [ ]:
t0 = time.time()
cat_table, _ = sbcat.category_table(S, seed=103)      # the same selection and seed as reproduce_daume_2024.ipynb
cat = cat_table[cat_table.is_category & cat_table.area.isin(sbcat.MTL)].reset_index(drop=True)
cat_trials = sbcat.trial_rates(S, cat)
print(f"category selection in {time.time() - t0:.0f} s:", cat.area.value_counts().to_dict())
h3_rows, pair_rows = [], []
for rd in READINGS:
    parts = []
    for i, u in cat.iterrows():
        s = S[u.session]
        keys = sbcat.bundle_keys(s["electrodes"])
        on = s["trials"].timestamps_Maintenance.to_numpy()
        sc_all = sbcat.count_in(s["spikes"][u.unit], on, on + 2.5)
        for ci, e in enumerate(s["lfp_electrodes"]):
            if keys[e] != keys[u.electrode] or (u.session, ci) not in chan_index:
                continue
            row = ch[(ch.reading == rd) & (ch.session == u.session) & (ch.channel == ci)].iloc[0]
            if not row.z_thetaHG > 1.64:
                continue
            c = chan_index[(u.session, ci)]
            idx = c["trial_index"]
            keep = c["masks"][rd][idx]
            st = c["st_thetaHG"][keep]
            parts.append(pd.DataFrame({"pac_a": st, "pac_b": (st - np.nanmean(st)) / np.nanstd(st), "sc": sc_all[idx][keep],
                                       "load1": (s["trials"].load.to_numpy()[idx][keep] == 1).astype(int),
                                       "pref": cat_trials[i]["pref"][idx][keep], "patient": u.subject,
                                       "pair": f"{u.session}_{u.unit}_{ci}", "area": u.area}))
    df3 = pd.concat(parts).dropna().reset_index(drop=True)
    for a in sbcat.MTL:
        da = df3[df3.area == a]
        pair_rows.append({"artefact": rd, "area": a, "pairs": da.pair.nunique(), "neurons": da.pair.str.rsplit("_", n=1).str[0].nunique(),
                          "patients": da.patient.nunique(), "trials": len(da), "spike count median": da.sc.median(), "max": da.sc.max()})
        for trials in ["all", "preferred"]:
            d = da[(trials == "all") | da.pref].reset_index(drop=True)
            for o5 in ["a", "b"]:
                try:
                    m = smf.mixedlm(f"pac_{o5} ~ sc * load1", d, groups="patient", vc_formula={"pair": "0 + C(pair)"}).fit(reml=True)
                    h3_rows.append({"artefact": rd, "area": a, "trials": trials, "O5": o5, "pairs": d.pair.nunique(), "n": len(d),
                                    "intercept": m.params["Intercept"], **slopes(m, "sc")})
                except Exception as e_:   # report a failed fit
                    h3_rows.append({"artefact": rd, "area": a, "trials": trials, "O5": o5, "error": str(e_)[:80]})
print(pd.DataFrame(pair_rows).to_string(index=False))
print("paper: 151 hippocampal and 423 amygdala neuron-channel combinations")
print(pd.DataFrame(h3_rows).to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("SC slope in load 3 trials (L3, the paper's SC term, load 3 the reference as in Table S3), in load 1 trials (L1) and averaged")
print("paper (Table S3): hippocampus intercept -0.044, SC 0.002, t = 2.40, p = 0.017, Load1 0.076, SC:Load1 -0.004 (p = 0.010),")
print("so its load 1 slope is 0.002 - 0.004 = -0.002; amygdala SC -0.001, t = -0.75, p = 0.45, SC:Load1 0.002 (p = 0.14)")

**Result: not reproduced (the verified outcome), with a change to reproduced proposed.** With
load 3 as the reference level, as Table S3 codes load, the SC term is the slope in load 3
trials. Over 166 hippocampal pairs (paper 151) and 450 amygdala pairs (paper 423), under artefact
reading (a), the hippocampal SC term is positive with p < 0.05 under every reading: all trials,
O5 (a) t = 6.34, p = 2.3e-10, and O5 (b) 0.0045, t = 3.09, p = 0.0020 (paper 0.002, t = 2.40, p
= 0.017); preferred trials, O5 (a) t = 3.09, p = 0.0020, and O5 (b) 0.0028, t = 2.16, p = 0.031.
The amygdala term has p >= 0.33 under every reading (paper p = 0.45). Under O5 (b) on all trials
the other terms are close to Table S3: intercept -0.057 (paper -0.044), Load1 0.078 (0.076),
SC x Load1 -0.0049, p = 0.0002 (-0.004, p = 0.010). The slope in load 1 trials, which the
earlier coding tested, is not positive (O5 b, all trials, p = 0.77; preferred, -0.0045, p =
0.015), as the paper's own load 1 slope (0.002 - 0.004) is not. Under artefact reading (b) (145
and 410 pairs) the hippocampal term is positive with p < 0.05 under three readings (p <= 0.014)
and has p = 0.056 under the fourth (preferred, O5 b); the amygdala term has p >= 0.32 (one of its
fits did not converge).

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| fig2a-theta-gamma-pac | reproduced |
| fig2b-pac-channel-proportions | partially reproduced (17 pre-SMA channels against 5) |
| fig2c-hippocampus-load | reproduced |
| fig2c-no-load-effect-other-areas | not reproduced (pre-SMA, and amygdala or dACC) |
| edf2b-hippocampus-load-patients | reproduced, resting on artefact reading (a) |
| edf2c-low-gamma-no-load-effect | not reproduced (pre-SMA) |
| tables1-slow-fast-theta | not reproduced (pre-SMA tests also significant) |
| fig2f-rt-single-trial-pac | not reproduced; change to blocked (methods underspecified) proposed |
| edf3a-kappa-load | reproduced |
| edf3e-nesting | reproduced, resting on the alignment on gamma peaks |
| edf3f-raw-mi-load | reproduced, resting on artefact reading (a) |
| methods-spectral-slope | not reproduced, resting on the 2 to 150 Hz range |
| fig3h-pac-category-firing | not reproduced; change to reproduced proposed |
| edf3-hippocampus-pac-controls | in `reproduce_daume_2024_lfp_controls.ipynb` |

How the analysis differs from the paper's: the artefact rejection is the automated part only
(the visual inspection cannot be repeated), under two readings of the span it is computed on;
reading (a) removes 11.4% of trials per channel and reading (b) 25.9%, against the paper's 14.6%
with visual inspection; where the readings disagree, (a) is adopted, as the paper's own rejected
trials on its example channel contain every trial (a) removes and only 4 of the 9 that (b)
removes. The single-trial PAC, the nesting alignment, the slope's frequency range and the power
estimate are not given by the paper. Mixed models are fitted with statsmodels in place of fitglme, with load 3 as the reference level as
in Tables S2 and S3; RT is fitted in s, as log10 RT and z-scored within session, as Table S2
gives no unit or transform.

What was found about the dataset: the LFP of 1,922 channels is there for 43 of the 44 sessions
(sub-20 has none), at 400 Hz in microvolts, and shares the trial clock (probe-locked contrast,
37 of 43 sessions). The files do not mark the trials or channels the paper's visual inspection
removed. Without them, the 17 significant pre-SMA channels, with a mean z of 6.87 in load 3
against 1.81 in load 1 under reading (a), decide several of the absence claims.

In [ ]:
print(f"notebook ran in {(time.time() - started) / 60:.1f} min")